# Sentinel — Evaluare extinsă v2

Rulează în Colab **sau** local.  
Necesită ca ml-service-ul Modal să fie deployed și accesibil la `ML_SERVICE_URL`.


In [ ]:
# ── 0. Configurare ────────────────────────────────────────────────────────────
# URL-ul aplicației Modal — format: https://<username>--<app-name>.modal.run
# Găsești URL-ul în dashboard Modal sau după `modal deploy`.
ML_SERVICE_URL = "http://localhost:8001"

HF_DATASET     = "EXANU/antiplagiator-artifacts"

# Număr de documente per nivel plagiat (mai mare = mai robust statistic)
N_DOCS_PER_LEVEL = 50
# Număr de documente pentru fals-pozitive
N_FALSE_POS = 30
# Seed pentru reproductibilitate
SEED = 42

In [8]:
import sys
print(sys.executable)
!{sys.executable} -m pip install requests huggingface_hub pandas matplotlib seaborn

c:\Users\Maxim Ernest\AppData\Local\Programs\Python\Python313\python.exe


'c:\Users\Maxim' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
# ── 2. Imports + warmup Modal ─────────────────────────────────────────────────
import json, random, time, math, statistics as st
from collections import defaultdict
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt

random.seed(SEED)

# ── Warmup: Modal poate fi in cold-start (containerul e oprit dupa inactivitate).
# Primul request poate dura 30-90s pana porneste. Asteptam cu retry.
def warmup_modal(url: str, max_wait: int = 120) -> bool:
    """Ping /health pana cand engine-ul e ready sau expira timeout-ul."""
    print(f"Conectare la {url} ...", end="", flush=True)
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            r = requests.get(f"{url}/health", timeout=30)
            if r.status_code == 200:
                data = r.json()
                if data.get("status") == "ready":
                    print(f" OK (engine ready)")
                    return True
                else:
                    print(".", end="", flush=True)
        except requests.exceptions.ConnectionError:
            print(".", end="", flush=True)   # cold start — containerul inca porneste
        except Exception as e:
            print(f" eroare: {e}")
        time.sleep(5)
    print(" TIMEOUT")
    return False

assert warmup_modal(ML_SERVICE_URL), "ML service inaccesibil — verifica URL-ul Modal"

health = requests.get(f"{ML_SERVICE_URL}/health", timeout=15).json()
print("Routing activ:", health.get("routing", {}))


def analyze(text: str, label: str = "", threshold: float = 0.85,
            paraphrase_mode: bool = False, top_k: int = 5) -> dict:
    """Trimite text la endpoint-ul /analyze_text si returneaza rezultatul."""
    resp = requests.post(
        f"{ML_SERVICE_URL}/analyze_text",
        json={"text": text, "label": label,
              "threshold": threshold, "top_k": top_k,
              "paraphrase_mode": paraphrase_mode},
        timeout=180,   # Modal poate fi lent la primul request dupa idle
    )
    resp.raise_for_status()
    return resp.json()


def score_from_result(r: dict) -> float:
    return r["result"]["global_plagiarism_score_percent"]

ModuleNotFoundError: No module named 'requests'

In [ ]:
# ── 3. Incarca chunked_database.jsonl de pe HuggingFace ──────────────────────
from huggingface_hub import hf_hub_download

db_path = hf_hub_download(
    repo_id=HF_DATASET, filename="chunked_database.jsonl",
    repo_type="dataset", token=HF_TOKEN or None,
    local_dir="/tmp/sentinel_eval"
)

chunks_by_cat   = defaultdict(list)   # cat -> [(arxiv_id, chunk_text), ...]
chunks_by_paper = defaultdict(list)   # arxiv_id -> [chunk_text, ...]
paper_cat       = {}                  # arxiv_id -> top_category

with open(db_path, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        aid  = d.get("arxiv_id", "")
        cat  = d.get("top_category", "Unknown")
        text = d.get("text", "").strip()
        if len(text.split()) < 50: continue
        chunks_by_cat[cat].append((aid, text))
        chunks_by_paper[aid].append(text)
        paper_cat[aid] = cat

all_cats = sorted(chunks_by_cat.keys())
print(f"Categorii disponibile: {len(all_cats)}")
for c in all_cats:
    print(f"  {c:<50} {len(chunks_by_cat[c]):>6} fragmente")

In [ ]:
# ── 4. Functii utilitare ──────────────────────────────────────────────────────

NEUTRAL_BASE = """This chapter presents the theoretical foundation and 
experimental methodology employed in the current study. The primary objective 
is to investigate the relationship between the observed phenomena and the 
underlying mathematical structure. Previous work in this area has established 
several important results that form the basis for our analysis. The remainder 
of this chapter is organized as follows. Section two describes the problem 
formulation. Section three outlines the experimental setup. Section four 
discusses the evaluation criteria. The proposed approach builds upon established 
techniques and introduces several novel contributions. We begin by reviewing 
the relevant background material before presenting our main results. The 
notation used throughout this work follows standard conventions in the field. 
All experiments were conducted under controlled conditions to ensure 
reproducibility. We report both quantitative metrics and qualitative analysis 
to provide a comprehensive evaluation of the proposed method."""

NEUTRAL_WORDS = NEUTRAL_BASE.split()

def build_synthetic_doc(copied_fragments: list, plagiarism_pct: float) -> str:
    """
    Construieste un document sintetic cu `plagiarism_pct` % text copiat.
    Restul e text neutru generic. Fragmentele sunt intercalate uniform.
    """
    copied_word_count = sum(len(f.split()) for f in copied_fragments)
    total_words       = int(copied_word_count / plagiarism_pct)
    neutral_words     = total_words - copied_word_count

    base_extended = (NEUTRAL_WORDS * (neutral_words // len(NEUTRAL_WORDS) + 2))[:neutral_words]

    step   = max(1, neutral_words // (len(copied_fragments) + 1))
    result = []
    pos    = 0
    for frag in copied_fragments:
        end = min(pos + step, len(base_extended))
        result.extend(base_extended[pos:end])
        result.extend(frag.split())
        pos = end
    result.extend(base_extended[pos:])
    return " ".join(result)


def simple_paraphrase(text: str) -> str:
    """
    Parafraza simpla prin substitutii lexicale. Nu necesita modele externe.
    Calibrat pentru vocabularul academic.
    """
    replacements = {
        "propose": "introduce", "proposed": "introduced",
        "method": "approach", "methods": "approaches",
        "show": "demonstrate", "shows": "demonstrates",
        "result": "outcome", "results": "outcomes",
        "use": "employ", "used": "employed", "using": "employing",
        "large": "substantial", "small": "limited",
        "important": "significant", "novel": "new",
        "obtain": "achieve", "obtained": "achieved",
        "analyze": "examine", "analysis": "examination",
        "present": "provide", "study": "investigation",
        "based on": "building upon", "in order to": "to",
        "furthermore": "additionally", "however": "nevertheless",
        "we": "the authors", "our": "the authors\'",
    }
    words = text.split()
    result = []
    i = 0
    while i < len(words):
        if i + 1 < len(words):
            bigram = (words[i] + " " + words[i+1]).lower()
            if bigram in replacements:
                result.append(replacements[bigram])
                i += 2
                continue
        result.append(replacements.get(words[i].lower(), words[i]))
        i += 1
    return " ".join(result)

In [ ]:
# ── SCENARIUL 1: Trei niveluri de plagiat (10%, 30%, 50%) ────────────────────
print("=" * 60)
print("SCENARIUL 1 — Detectie copiere directa: 10% / 30% / 50%")
print("=" * 60)

levels     = [0.10, 0.30, 0.50]
frags_map  = {0.10: 1, 0.30: 3, 0.50: 5}

all_papers = [aid for aid, chunks in chunks_by_paper.items() if len(chunks) >= 5]
random.shuffle(all_papers)
test_papers = all_papers[:N_DOCS_PER_LEVEL]

results_s1 = []

for pct in levels:
    n_frags     = frags_map[pct]
    found_first = 0
    found_top3  = 0
    scores      = []
    times       = []

    for aid in test_papers:
        paper_chunks = chunks_by_paper[aid]
        mid   = len(paper_chunks) // 2
        frags = paper_chunks[mid : mid + n_frags]
        if len(frags) < n_frags:
            frags = paper_chunks[:n_frags]

        doc_text = build_synthetic_doc(frags, pct)

        t0  = time.time()
        res = analyze(doc_text, label=f"s1_{int(pct*100)}pct_{aid}")
        dt  = time.time() - t0

        score     = score_from_result(res)
        sources   = res["result"].get("sources", [])
        found_ids = [s["arxiv_id"] for s in sources]

        scores.append(score)
        times.append(dt)
        if found_ids and found_ids[0] == aid: found_first += 1
        if aid in found_ids[:3]:              found_top3  += 1

        print(f"  [{int(pct*100)}%] {aid} -> score={score:.1f}% "
              f"first={'V' if found_ids and found_ids[0]==aid else 'X'} "
              f"top3={'V' if aid in found_ids[:3] else 'X'}")

    n = len(test_papers)
    results_s1.append({
        "Nivel plagiat": f"{int(pct*100)}%",
        "Sursa corecta (locul 1)": f"{found_first/n*100:.1f}%",
        "Sursa corecta (top 3)": f"{found_top3/n*100:.1f}%",
        "Scor mediu raportat": f"{st.mean(scores):.1f}%",
        "Abatere standard": f"{st.stdev(scores):.1f}%",
        "Durata medie (s)": f"{st.mean(times):.1f}",
    })
    print(f"  -> Nivel {int(pct*100)}%: primul={found_first}/{n}, top3={found_top3}/{n}, "
          f"scor mediu={st.mean(scores):.1f}%")

df_s1 = pd.DataFrame(results_s1)
print("\nTabel Scenariul 1:")
print(df_s1.to_string(index=False))

In [ ]:
# ── SCENARIUL 2: Fals-pozitive per categorie ──────────────────────────────────
print("=" * 60)
print("SCENARIUL 2 — Fals-pozitive: distributie per categorie")
print("=" * 60)
print("Nota: folosim texte neutre sintetice (fara fragmente din corpus).")
print("Pentru FP adeverate, inlocuieste cu arxiv papers post corpus build date.\n")

results_s2_exact      = defaultdict(list)
results_s2_paraphrase = defaultdict(list)

n_per_cat = max(2, N_FALSE_POS // len(all_cats))

for cat in all_cats:
    for i in range(n_per_cat):
        neutral_doc = " ".join(NEUTRAL_WORDS * 10)  # ~2000 cuvinte

        res_e = analyze(neutral_doc, label=f"s2_fp_exact_{cat}_{i}")
        results_s2_exact[cat].append(score_from_result(res_e))

        res_p = analyze(neutral_doc, label=f"s2_fp_para_{cat}_{i}",
                        paraphrase_mode=True)
        results_s2_paraphrase[cat].append(score_from_result(res_p))

    se = results_s2_exact[cat]
    sp = results_s2_paraphrase[cat]
    print(f"  {cat[:45]:<45} exact={st.mean(se):.1f}%  para={st.mean(sp):.1f}%")

rows_s2 = []
for cat in all_cats:
    se = results_s2_exact[cat]
    sp = results_s2_paraphrase[cat]
    rows_s2.append({
        "Categorie": cat,
        "FP exact (medie)": f"{st.mean(se):.1f}%",
        "FP exact (max)": f"{max(se):.1f}%",
        "FP parafraze (medie)": f"{st.mean(sp):.1f}%",
        "FP parafraze (max)": f"{max(sp):.1f}%",
    })

df_s2 = pd.DataFrame(rows_s2).sort_values("FP exact (medie)", ascending=False)
print("\nTabel Scenariul 2 (fals-pozitive per categorie):")
print(df_s2.to_string(index=False))

In [ ]:
# ── SCENARIUL 3: Parafraze vs copiere directa ─────────────────────────────────
print("=" * 60)
print("SCENARIUL 3 — Detectie parafraze vs copiere directa")
print("=" * 60)

test_papers_s3 = random.sample(all_papers, min(N_DOCS_PER_LEVEL, len(all_papers)))
results_s3 = []

for aid in test_papers_s3:
    paper_chunks = chunks_by_paper[aid]
    if len(paper_chunks) < 3: continue

    mid   = len(paper_chunks) // 2
    frags = paper_chunks[mid : mid + 3]

    doc_exact  = build_synthetic_doc(frags, 0.30)
    frags_para = [simple_paraphrase(f) for f in frags]
    doc_para   = build_synthetic_doc(frags_para, 0.30)

    r_ee = analyze(doc_exact, label=f"s3_exact_strict_{aid}")
    r_ep = analyze(doc_exact, label=f"s3_exact_para_{aid}", paraphrase_mode=True)
    r_pe = analyze(doc_para,  label=f"s3_para_strict_{aid}")
    r_pp = analyze(doc_para,  label=f"s3_para_para_{aid}", paraphrase_mode=True)

    results_s3.append({
        "arxiv_id":              aid,
        "categorie":             paper_cat.get(aid, "?"),
        "exact_mod_strict":      score_from_result(r_ee),
        "exact_mod_parafraze":   score_from_result(r_ep),
        "parafraza_mod_strict":  score_from_result(r_pe),
        "parafraza_mod_parafraze": score_from_result(r_pp),
    })
    row = results_s3[-1]
    print(f"  {aid} [{row['categorie'][:18]:<18}] "
          f"exact/strict={row['exact_mod_strict']:.1f}% "
          f"para/para={row['parafraza_mod_parafraze']:.1f}%")

df_s3 = pd.DataFrame(results_s3)

summary_s3 = pd.DataFrame([
    {"Varianta document": "Copiere directa",
     "Mod strict":    f"{st.mean(df_s3.exact_mod_strict):.1f}% +/- {st.stdev(df_s3.exact_mod_strict):.1f}%",
     "Mod parafraze": f"{st.mean(df_s3.exact_mod_parafraze):.1f}% +/- {st.stdev(df_s3.exact_mod_parafraze):.1f}%"},
    {"Varianta document": "Parafrazare simpla",
     "Mod strict":    f"{st.mean(df_s3.parafraza_mod_strict):.1f}% +/- {st.stdev(df_s3.parafraza_mod_strict):.1f}%",
     "Mod parafraze": f"{st.mean(df_s3.parafraza_mod_parafraze):.1f}% +/- {st.stdev(df_s3.parafraza_mod_parafraze):.1f}%"},
])
print("\nRezumate Scenariul 3:")
print(summary_s3.to_string(index=False))

extra_detected = sum(
    1 for r in results_s3
    if r["parafraza_mod_parafraze"] > 5.0 and r["parafraza_mod_strict"] < 2.0
)
print(f"\nCazuri detectate EXCLUSIV de modul parafraze: {extra_detected}/{len(results_s3)}")

In [ ]:
# ── VIZUALIZARI ───────────────────────────────────────────────────────────────
import os
os.makedirs("/tmp/sentinel_eval", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graf 1: Scor mediu vs nivel plagiat
ax1 = axes[0]
lvl_labels  = [r["Nivel plagiat"] for r in results_s1]
scores_med  = [float(r["Scor mediu raportat"].strip("%")) for r in results_s1]
scores_std  = [float(r["Abatere standard"].strip("%")) for r in results_s1]
ax1.bar(lvl_labels, scores_med, color="#378ADD", yerr=scores_std, capsize=5, alpha=0.85)
for i, t in enumerate([int(l.strip("%")) for l in lvl_labels]):
    ax1.axhline(t, xmin=i/3+0.02, xmax=(i+1)/3-0.02, linestyle="--", color="gray", linewidth=0.8)
ax1.set_xlabel("Nivel plagiat")
ax1.set_ylabel("Scor raportat (%)")
ax1.set_title("Scor raportat vs nivel plagiat real")
ax1.grid(axis="y", alpha=0.3)

# Graf 2: Fals-pozitive per categorie
ax2 = axes[1]
cats_short = [
    c.replace("High Energy Physics - ", "HEP-")
     .replace("General Relativity and Quantum Cosmology", "GR&QC")
     .replace("Electrical Engineering and Systems Science", "EESS")
     .replace("Quantitative ", "Q.")
    for c in df_s2["Categorie"]
]
fp_exact = [float(v.strip("%")) for v in df_s2["FP exact (medie)"]]
fp_para  = [float(v.strip("%")) for v in df_s2["FP parafraze (medie)"]]
y = range(len(cats_short))
ax2.barh([i + 0.2 for i in y], fp_para,  height=0.4, color="#D85A30", label="Mod parafraze", alpha=0.85)
ax2.barh([i - 0.2 for i in y], fp_exact, height=0.4, color="#378ADD", label="Mod strict",    alpha=0.85)
ax2.set_yticks(list(y))
ax2.set_yticklabels(cats_short, fontsize=8)
ax2.set_xlabel("Scor fals-pozitiv mediu (%)")
ax2.set_title("Fals-pozitive per categorie")
ax2.legend()
ax2.grid(axis="x", alpha=0.3)

# Graf 3: Exact vs parafraze — comparatie moduri
ax3 = axes[2]
labels3 = ["Copiere directa\nmod strict", "Copiere directa\nmod parafraze",
            "Parafrazare\nmod strict", "Parafrazare\nmod parafraze"]
cols3   = ["exact_mod_strict", "exact_mod_parafraze",
            "parafraza_mod_strict", "parafraza_mod_parafraze"]
means3  = [st.mean(df_s3[c]) for c in cols3]
stds3   = [st.stdev(df_s3[c]) for c in cols3]
colors3 = ["#378ADD", "#1D9E75", "#378ADD", "#1D9E75"]
ax3.bar(labels3, means3, color=colors3, yerr=stds3, capsize=5, alpha=0.85)
ax3.set_ylabel("Scor raportat (%)")
ax3.set_title("Impact mod de detectie x tip document")
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/sentinel_eval/evaluation_extended_v2.png", dpi=150)
plt.show()
print("Figuri salvate.")

In [ ]:
# ── Export LaTeX automat ──────────────────────────────────────────────────────
def df_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:
    cols    = " & ".join(df.columns) + " \\\\"
    rows    = [" & ".join(str(v) for v in row) + " \\\\" for row in df.values]
    body    = "\n        ".join(rows)
    col_fmt = "l" + "r" * (len(df.columns) - 1)
    return (
        f"\\begin{{table}}[ht]\n"
        f"\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"\\small\n"
        f"\\begin{{tabular}}{{{col_fmt}}}\n"
        f"    \\toprule\n"
        f"    {cols}\n"
        f"    \\midrule\n"
        f"    {body}\n"
        f"    \\bottomrule\n"
        f"\\end{{tabular}}\n"
        f"\\end{{table}}"
    )

print(df_to_latex(df_s1, "Rezultatele detectiei la trei niveluri de plagiat.", "tab:eval-levels"))
print()
print(df_to_latex(summary_s3, "Comparatie mod strict vs mod parafraze.", "tab:eval-para"))

In [ ]:
# ── Download outputs (Colab) ──────────────────────────────────────────────────
df_s3.to_csv("/tmp/sentinel_eval/results_s3_raw.csv", index=False)
try:
    from google.colab import files
    files.download("/tmp/sentinel_eval/evaluation_extended_v2.png")
    files.download("/tmp/sentinel_eval/results_s3_raw.csv")
except ImportError:
    print("Nu rulezi in Colab — fisierele sunt in /tmp/sentinel_eval/")